In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install tqdm boto3 requests regex sentencepiece sacremoses --quiet
!pip install transformers --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.2/83.2 kB 6.7 MB/s eta 0:00:00


In [ ]:
from tqdm.auto import tqdm
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
import re

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import transformers
from transformers import BertModel, BertTokenizer, get_scheduler, set_seed, AutoTokenizer, AutoModel
from sklearn.metrics import classification_report, confusion_matrix
from gensim.utils import simple_preprocess

In [ ]:
# CONFIGURATION CLASS
@dataclass
class Config:
    batch_size: int = 16
    pin_memory: bool = True
    num_workers: int = 2
    seed: int = 42
    lr: float = 5e-5

In [ ]:
config = Config()
set_seed(config.seed)

In [ ]:
with open('/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Train.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_train = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_train = df_train.drop(['id'], axis=1)

with open('/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Test.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_test = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_test = df_test.drop(['id'], axis=1)

with open('/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Dev.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_val = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_val = df_val.drop(['id'], axis=1)

In [ ]:
# Define the mappings
aspect_to_index = {
    'restaurant#general': 0, 'restaurant#prices': 1, 'restaurant#miscellaneous': 2, 'food#quality': 3,
    'food#prices': 4, 'food#style&option': 5, 'drinks#quality': 6, 'drinks#prices': 7,
    'drinks#style&option': 8, 'location#general': 9, 'ambience#general': 10, 'service#general': 11
}
# Adding 'None' to handle unspecified sentiments for any detected aspect
sentiment_to_index = {'positive': 1, 'negative': 0, 'neutral': 2, 'none': 3}

# Preprocess the labels
def preprocess_labels(label_str):
    labels = re.findall(r'\{(.*?)\}', label_str.lower())
    aspects = torch.zeros(len(aspect_to_index), dtype=torch.long)
    sentiments = torch.full((len(aspect_to_index),), sentiment_to_index['none'], dtype=torch.long)

    # Initialize with 'None'
    for label in labels:
        if ', ' in label:
            aspect, sentiment = label.split(', ')
            if aspect in aspect_to_index and sentiment in sentiment_to_index:  # Only process known aspects with sentiments
                idx = aspect_to_index[aspect]
                aspects[idx] = 1
                sentiments[idx] = sentiment_to_index[sentiment]

    return aspects, sentiments

df_train['aspects'], df_train['sentiments'] = zip(*df_train['label'].apply(preprocess_labels))
df_test['aspects'], df_test['sentiments'] = zip(*df_test['label'].apply(preprocess_labels))
df_val['aspects'], df_val['sentiments'] = zip(*df_val['label'].apply(preprocess_labels))

In [ ]:
df_train

,text,label,aspects,sentiments
0,Giá 53k size vừa.,"{DRINKS#PRICES, neutral}, {DRINKS#STYLE&OPTION...","[tensor(0), tensor(0), tensor(0), tensor(0), t...","[tensor(3), tensor(3), tensor(3), tensor(3), t..."
1,Nhưng nói chung cũng hơi đắt.,"{RESTAURANT#PRICES, negative}","[tensor(0), tensor(1), tensor(0), tensor(0), t...","[tensor(3), tensor(0), tensor(3), tensor(3), t..."
2,Mình ăn rất hôi mùi dầu.,"{FOOD#QUALITY, negative}","[tensor(0), tensor(0), tensor(0), tensor(1), t...","[tensor(3), tensor(3), tensor(3), tensor(0), t..."
3,Mình ăn chưa baoh thấy mùi hôi hải sản.,"{FOOD#QUALITY, positive}","[tensor(0), tensor(0), tensor(0), tensor(1), t...","[tensor(3), tensor(3), tensor(3), tensor(1), t..."
4,3 dĩa vs 2 lon Revive mà có 190k thui(.,"{RESTAURANT#PRICES, positive}","[tensor(0), tensor(1), tensor(0), tensor(0), t...","[tensor(3), tensor(1), tensor(3), tensor(3), t..."
...,...,...,...,...
7023,Cơ mà hôm đó bạn nv order có vẻ khó chịu với m...,"{SERVICE#GENERAL, negative}","[tensor(0), tensor(0), tensor(0), tensor(0), t...","[tensor(3), tensor(3), tensor(3), tensor(3), t..."
7024,Tổng hoá đơn phần bò sốt là 115k.,"{FOOD#PRICES, neutral}","[tensor(0), tensor(0), tensor(0), tensor(0), t...","[tensor(3), tensor(3), tensor(3), tensor(3), t..."
7025,Quán trang trí bắt mắt và sáng sủa.,"{AMBIENCE#GENERAL, positive}","[tensor(0), tensor(0), tensor(0), tensor(0), t...","[tensor(3), tensor(3), tensor(3), tensor(3), t..."
7026,"Trà đào ở đây đậm đà hơn, ngon hơn mấy chỗ khá...","{DRINKS#QUALITY, positive}, {RESTAURANT#MISCEL...","[tensor(0), tensor(0), tensor(1), tensor(0), t...","[tensor(3), tensor(3), tensor(2), tensor(3), t..."


In [ ]:
from bs4 import BeautifulSoup

for df in [df_train, df_test, df_val]:
  for sentence in range(0, len(df.text)):
    # xóa tag, link http
    processed_feature = BeautifulSoup(str(df.text[sentence]), "html.parser").get_text()
    #email-id
    processed_feature = re.sub('b[w-]+?@w+?.w{2,4}b', 'emailadd', processed_feature)
    #url
    processed_feature = re.sub('(http[s]?S+)|(w+.[A-Za-z]{2,4}S*)', 'urladd', processed_feature)

    # Xóa tất cả các ký tự đặc biệt
    processed_feature = re.sub(r'\W', ' ', processed_feature)

    # xóa từ có chứa chữ số
    # processed_feature = re.sub('W*dw*', '', processed_feature)

    # Chuyển đổi sang chữ thường
    processed_feature = processed_feature.lower()

    # Thay thế nhiều khoảng trắng bằng một khoảng trắng
    processed_feature = re.sub(r'\s+', ' ', processed_feature, flags=re.I)

    # processed_features.append(processed_feature)
    df.text[sentence] = processed_feature

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df.text[sentence] = processed_feature
<ipython-input-9-11490973c74e>:25: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] =

In [ ]:
df_train['aspects'] = df_train['aspects'].apply(lambda x: x.numpy())
df_train['sentiments'] = df_train['sentiments'].apply(lambda x: x.numpy())

In [ ]:
df_test['aspects'] = df_test['aspects'].apply(lambda x: x.numpy())
df_test['sentiments'] = df_test['sentiments'].apply(lambda x: x.numpy())

In [ ]:
df_val['aspects'] = df_val['aspects'].apply(lambda x: x.numpy())
df_val['sentiments'] = df_val['sentiments'].apply(lambda x: x.numpy())

In [ ]:
import gensim

documents = [_text.split() for _text in df_train.text]
w2v_model = gensim.models.word2vec.Word2Vec(vector_size=128,
                                            window=7,
                                            min_count=10,
                                            workers=8)

w2v_model.build_vocab(documents)

w2v_model.train(documents, total_examples=len(documents), epochs=32)

(2701222, 3666912)

In [ ]:
words = w2v_model.wv
vocab_size = len(words)
print("Vocab size", vocab_size)

Vocab size 1147


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df_train.text)

vocab_size = len(tokenizer.word_index) + 1
print("Total words", vocab_size)

Total words 4491


In [ ]:
embedding_matrix = np.zeros((vocab_size, 128))
for word, i in tokenizer.word_index.items():
  if word in w2v_model.wv:
    embedding_matrix[i] = w2v_model.wv[word]
print(embedding_matrix.shape)

(4491, 128)


In [ ]:
X_train = pad_sequences(tokenizer.texts_to_sequences(df_train.text.values), maxlen=128)
X_test = pad_sequences(tokenizer.texts_to_sequences(df_test.text.values), maxlen=128)
X_val = pad_sequences(tokenizer.texts_to_sequences(df_val.text.values), maxlen=128)

In [ ]:
y_train = np.stack(df_train.aspects.values)
y_test = np.stack(df_test.aspects.values)
y_val = np.stack(df_val.aspects.values)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, Dense, Embedding, LSTM, Dropout, Conv1D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.activations import gelu
from tensorflow.keras.optimizers import AdamW

embedding = Embedding(input_dim=vocab_size,
                      output_dim=128,
                      weights=[embedding_matrix],
                      trainable=True)
# model.summary()
callbacks = EarlyStopping(monitor='val_loss', min_delta=1e-4, patience=5)

In [ ]:
modelLSTM = Sequential()
modelLSTM.add(embedding)
modelLSTM.add(LSTM(256, return_sequences=True, input_shape=(X_train.shape[1],1)))
modelLSTM.add(LSTM(256, return_sequences=True))
modelLSTM.add(LSTM(256))
modelLSTM.add(Dropout(0.3))
modelLSTM.add(Dense(12, activation=gelu))
adamw = AdamW(learning_rate=5e-5, weight_decay=0.0001)
modelLSTM.compile(loss='categorical_crossentropy',
              optimizer=adamw,
              metrics=['accuracy'])



/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
historyLSTM = modelLSTM.fit(X_train, y_train,
                            batch_size=16,
                            epochs=100,
                            verbose=1,
                            validation_data=(X_val, y_val),)
                            # callbacks= callbacks)

Epoch 1/100
440/440 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.1557 - loss: 7.4522 - val_accuracy: 0.1297 - val_loss: 6.2035
Epoch 2/100
440/440 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - accuracy: 0.1614 - loss: 6.3914 - val_accuracy: 0.2672 - val_loss: 7.9836
Epoch 3/100
440/440 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - accuracy: 0.2682 - loss: 7.4622 - val_accuracy: 0.3100 - val_loss: 8.0131
Epoch 4/100
440/440 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 0.2890 - loss: 8.0143 - val_accuracy: 0.2685 - val_loss: 7.6833
Epoch 5/100
440/440 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 0.2462 - loss: 8.3407 - val_accuracy: 0.2866 - val_loss: 8.9334
Epoch 6/100
440/440 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 0.2609 - loss: 8.0971 - val_accuracy: 0.2892 - val_loss: 9.6651
Epoch 7/100
440/440 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 0.2675 - loss: 8.7347 - val_accuracy: 0.3087 - val_loss: 7.9386
Epoch 8/100
440/440 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 0.2900 - loss: 8

In [ ]:
score = modelLSTM.evaluate(X_test, y_test, batch_size=16)
print()
print("ACCURACY:",score[1])
print("LOSS:",score[0])

122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.2492 - loss: 8.0173

ACCURACY: 0.24819400906562805
LOSS: 8.237651824951172


In [ ]:
sentence = "Nhà hàng có không gian đẹp, nhưng chất lượng món ăn lại không như mong đợi, phục vụ hơi chậm."
input = pad_sequences(tokenizer.texts_to_sequences([sentence]), maxlen=128)
prediction = modelLSTM.predict(input)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
[[ 0.1396619  -0.04811819  0.07205793  0.22914422 -0.11080883 -0.15541601
  -0.15679461 -0.16880886 -0.16043197  0.1128915   0.3054518   0.15645948]]


In [ ]:
max(prediction[0])

0.3054518

In [ ]:
def plot(history, name="HistoryPlot", figsize=(20, 9)):
    fig = plt.figure(figsize=figsize)
    epochs = [x['epoch'] for x in history]

    # Plotting Losses
    ax1 = fig.add_subplot(121)
    ax1.locator_params(axis='x', nbins=5)
    train_losses = [x['train_loss'] for x in history]
    val_losses = [x['val_loss'] for x in history]
    ax1.plot(epochs, train_losses, label='Train-Losses')
    ax1.plot(epochs, val_losses, label='Validation-Losses')
    plt.xlabel('Epochs')
    plt.ylabel('Losses')
    plt.title('Losses vs Epochs')
    plt.legend()

    # Plotting Accuracies
    ax2 = fig.add_subplot(122)
    ax2.locator_params(axis='x', nbins=5)
    train_accs = [x['train_acc'].cpu() for x in history]
    val_accs = [x['val_acc'].cpu() for x in history]
    ax2.plot(epochs, train_accs, label='Train-Accuracies')
    ax2.plot(epochs, val_accs, label='Validation-Accuracies')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracies')
    plt.title('Accuracies vs Epochs')
    plt.legend()

    fig.savefig('./'+name+".jpg")
    plt.show()

In [ ]:
plot(historyLSTM, name="HistoryPlot1")

TypeError: 'History' object is not iterable

<Figure size 2000x900 with 0 Axes>